# DeepFilterNet4 `train_dynamic` (Google Colab)

This notebook prepares a Colab runtime and launches `df_mlx.train_dynamic` with:

- run config: `DeepFilterNet/df_mlx/configs/run_profiles/pipeline_awesome_gan_curriculum_speech_only.toml`
- cache override via `--cache-hf` (default) or `--cache-dir`

Why cache override is required: the run config points at a local macOS path, so Colab must supply its own datastore source.

Set `SMOKE_TEST = False` for full training behavior from the profile.


In [ ]:
import importlib.util
import os
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not running in Colab; skipping Google Drive mount.")

REPO_URL = "https://github.com/sealad886/DeepFilterNet4.git"
REPO_BRANCH = "main"

# Cache source: pick exactly one
CACHE_HF_REPO = "sealad886/dfn4_mlx_datastore"
CACHE_DIR = ""  # Example: /content/drive/MyDrive/DeepFilterNet/mlx_datastore

# Optional HF token (needed for private datasets / higher rate limits)
HF_TOKEN = ""

# Runtime behavior
SMOKE_TEST = True
SMOKE_EPOCHS = 1
SMOKE_MAX_TRAIN_BATCHES = 8
SMOKE_MAX_VALID_BATCHES = 2

# Extra CLI flags appended to the train command
EXTRA_FLAGS = ""

REPO_DIR = Path("/content/DeepFilterNet") if IN_COLAB else Path.cwd().resolve()
PROJECT_DIR = REPO_DIR / "DeepFilterNet"
RUN_CONFIG_PATH = PROJECT_DIR / "df_mlx" / "configs" / "run_profiles" / "pipeline_awesome_gan_curriculum_speech_only.toml"
VENV_DIR = REPO_DIR / ".venv"
VENV_PY = VENV_DIR / "bin" / "python"
CHECKPOINT_DIR = (
    Path("/content/drive/MyDrive/DeepFilterNet/checkpoints/gan_curriculum_speech_only")
    if IN_COLAB
    else REPO_DIR / "checkpoints" / "gan_curriculum_speech_only"
)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("IN_COLAB:", IN_COLAB)
print("REPO_DIR:", REPO_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("RUN_CONFIG_PATH:", RUN_CONFIG_PATH)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)


In [ ]:
%%bash
set -euo pipefail

if [[ -d /content ]]; then
  if [[ ! -d /content/DeepFilterNet/.git ]]; then
    git clone --depth 1 --branch main https://github.com/sealad886/DeepFilterNet4.git /content/DeepFilterNet
  else
    git -C /content/DeepFilterNet fetch --all --prune
    git -C /content/DeepFilterNet checkout main
    git -C /content/DeepFilterNet pull --ff-only
  fi

  apt-get update -y
  apt-get install -y \
    git ffmpeg libsndfile1 libsndfile1-dev \
    build-essential pkg-config cmake \
    rustc cargo python3-venv python3-dev \
    libportaudio2 portaudio19-dev
else
  echo "Non-Colab runtime detected; skipping clone + apt bootstrap."
fi


In [ ]:
%%bash
set -euo pipefail

REPO_DIR="/content/DeepFilterNet"
if [[ ! -d /content ]]; then
  REPO_DIR="$(pwd)"
fi

VENV_DIR="${REPO_DIR}/.venv"
VENV_PY="${VENV_DIR}/bin/python"

python3 -m venv "${VENV_DIR}"

"${VENV_PY}" -m pip install --upgrade pip setuptools wheel maturin

# Build local Rust-backed libs first (required by DeepFilterNet package).
"${VENV_PY}" -m pip install -e "${REPO_DIR}/pyDF"
"${VENV_PY}" -m pip install -e "${REPO_DIR}/pyDF-data"

# Install Python package + train/eval extras needed by train_dynamic profile.
"${VENV_PY}" -m pip install -e "${REPO_DIR}/DeepFilterNet[train,eval]"

# Install MLX backend per runtime.
if command -v nvidia-smi >/dev/null 2>&1; then
  if ! "${VENV_PY}" -m pip install --upgrade "mlx[cuda]"; then
    "${VENV_PY}" -m pip install --upgrade "mlx[cuda12]"
  fi
else
  "${VENV_PY}" -m pip install --upgrade "mlx[cpu]"
fi

# Silero VAD path in this run-config requires onnxruntime + silero-vad.
"${VENV_PY}" -m pip install --upgrade onnxruntime silero-vad

cd "${REPO_DIR}/DeepFilterNet"
"${VENV_PY}" - <<'PY'
import mlx
import df_mlx.train_dynamic
import df_mlx.vad_silero
print("Environment import check passed.")
PY


In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

if not VENV_PY.exists():
    raise FileNotFoundError(f"Python interpreter not found: {VENV_PY}")
if not RUN_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Run config not found: {RUN_CONFIG_PATH}")

cache_hf = (CACHE_HF_REPO or "").strip()
cache_dir = (CACHE_DIR or "").strip()
if bool(cache_hf) == bool(cache_dir):
    raise ValueError("Set exactly one of CACHE_HF_REPO or CACHE_DIR.")

cmd = [
    str(VENV_PY),
    "-m",
    "df_mlx.train_dynamic",
    "--run-config",
    str(RUN_CONFIG_PATH),
    "--checkpoint-dir",
    str(CHECKPOINT_DIR),
]

if cache_hf:
    normalized = cache_hf.removeprefix("hf://").removeprefix("datasets/")
    cmd += ["--cache-hf", normalized]
else:
    cmd += ["--cache-dir", cache_dir]

if SMOKE_TEST:
    cmd += [
        "--epochs",
        str(SMOKE_EPOCHS),
        "--max-train-batches",
        str(SMOKE_MAX_TRAIN_BATCHES),
        "--max-valid-batches",
        str(SMOKE_MAX_VALID_BATCHES),
        "--validate-every",
        "1",
    ]

if EXTRA_FLAGS.strip():
    cmd += shlex.split(EXTRA_FLAGS)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["DFNET_TQDM"] = "1"
if HF_TOKEN.strip():
    env["HF_TOKEN"] = HF_TOKEN.strip()

print("Training command:")
print(" ".join(shlex.quote(part) for part in cmd))

subprocess.run(cmd, cwd=str(PROJECT_DIR), env=env, check=True)


In [ ]:
from pathlib import Path

ckpt_dir = Path(CHECKPOINT_DIR)
if not ckpt_dir.exists():
    print(f"Checkpoint directory not found: {ckpt_dir}")
else:
    artifacts = sorted(ckpt_dir.glob("*"), key=lambda p: p.stat().st_mtime, reverse=True)
    print(f"Checkpoint artifacts in {ckpt_dir} (latest first):")
    for p in artifacts[:30]:
        print(p.name)
